# Prepare Additional Test Sets (mtcsd, ezstance)

**Goal:** check whether the semeval-trained model generalizes beyond SemEval's own
domains -- push mtcsd's and EZ-STANCE's held-out **test** splits to the HF Hub (same
`conversations` format as `04_prepare_test_set.ipynb`) so Colab can evaluate against
them too.

**These are eval sets, not training data** -- and specifically their **test** splits,
not `mtcsd_train.csv` (which notebook 02 already used as a training source for the
combined dataset). Using the test splits here means no overlap with anything the model
was or could be trained on.

Self-contained on purpose, same as the other notebooks: prompt/label logic defined
inline below, not imported from `prepare_stance_data.py`.


## 1. Define the prompt template + label map

Identical to every other notebook -- must match exactly, since this is what the
fine-tuned model is actually evaluated against.


In [1]:
STANCE_PROMPT_TEMPLATE = (
    "Stance classification is the task of determining the expressed or implied opinion, "
    "or stance, of a document toward a certain, specified target. "
    "Analyze the following document and determine its stance toward the provided query.\n\n"
    "QUERY: {target}\n\n"
    "DOCUMENT: {text}\n\n"
    'Return valid JSON in exactly this format: {{"stance": "FAVOR"}}\n'
    'The "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\n'
    'Use "FAVOR" only when the author is definitely in favor of the query. '
    'Use "AGAINST" only when the author is definitely against the query. '
    'Use "NONE" if any of the following holds: (a) the document does not discuss the query '
    "at all, (b) the document discusses it but the author takes no clear side "
    "(neutral/balanced), or (c) the author's position cannot be determined with confidence. "
    "Do not guess from indirect hints.\n"
)

# Maps every label spelling seen across stance datasets onto our fixed 3-way vocabulary.
STANCE_LABEL_MAP = {
    "FAVOR": "FAVOR",
    "AGAINST": "AGAINST",
    "NONE": "NONE",
    "PRO": "FAVOR",
    "NEUTRAL": "NONE",
    "UNCLEAR": "NONE",
    "UNRELATED": "NONE",
    "SUPPORTS": "FAVOR",
    "DENIES": "AGAINST",
}


def canonicalize_label(raw_label):
    normalized = str(raw_label).strip().upper()
    if normalized not in STANCE_LABEL_MAP:
        raise ValueError(
            f"Unrecognized stance label {raw_label!r} -- add it to STANCE_LABEL_MAP "
            f"(known: {sorted(STANCE_LABEL_MAP)})"
        )
    return STANCE_LABEL_MAP[normalized]


def build_conversations(df, text_column, target_column, label_column):
    conversations = []
    for _, row in df.iterrows():
        prompt = STANCE_PROMPT_TEMPLATE.format(target=row[target_column], text=row[text_column])
        answer = json.dumps({"stance": canonicalize_label(row[label_column])})
        conversations.append(
            [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer},
            ]
        )
    return conversations


## 2. Define the test-set sources

Both files already share the same schema (`content`/`query`/`stance_label`) as
`mtcsd_train.csv` from notebook 02 -- no new column mapping needed.

`ezstance_test_mixed.csv` (not `ezstance_test.csv`) is used because it's the one
already referenced in the existing Tilburg eval outputs
(`data_out/stance/ezstance_old_mixed/ezstance_test_mixed/...`).


In [2]:
from pathlib import Path

import pandas as pd

TEST_SOURCES = [
    {
        "name": "mtcsd",
        "path": Path(
            "/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper"
            "/data_in/mtcsd/mtcsd_test.csv"
        ),
        "push_to_hub_id": "nityaak/mtcsd-stance-test",
    },
    {
        "name": "ezstance",
        "path": Path(
            "/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper"
            "/data_in/ezstance/ezstance_test_mixed.csv"
        ),
        "push_to_hub_id": "nityaak/ezstance-stance-test-mixed",
    },
]


## 3. Load, build conversations, and check label balance per source


In [4]:
from datasets import Dataset
import json
test_datasets = {}

for src in TEST_SOURCES:
    df = pd.read_csv(src["path"])
    conversations = build_conversations(
        df, text_column="content", target_column="query", label_column="stance_label"
    )
    test_datasets[src["name"]] = Dataset.from_dict({"conversations": conversations})
    print(f"{src['name']}: {len(conversations)} examples")
    print(df["stance_label"].value_counts())
    print()


mtcsd: 2371 examples
stance_label
none       1181
against     724
favor       466
Name: count, dtype: int64

ezstance: 7798 examples
stance_label
favor      2669
none       2653
against    2476
Name: count, dtype: int64



## 4. Push each to its own HF Hub repo

Separate repos, not combined -- these are two distinct generalization checks, not
meant to be merged into one dataset.


In [5]:
PRIVATE = True

# Uncomment when ready:
for src in TEST_SOURCES:
    test_datasets[src["name"]].push_to_hub(src["push_to_hub_id"], private=False)
    print(f"Pushed {src['name']} to https://huggingface.co/datasets/{src['push_to_hub_id']}")


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 209.55ba/s]
Processing Files (1 / 1): 100%|██████████|  443kB /  443kB, 39.7kB/s  
New Data Upload: 100%|██████████|  443kB /  443kB, 39.7kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.23s/ shards]


Pushed mtcsd to https://huggingface.co/datasets/nityaak/mtcsd-stance-test


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 62.82ba/s]
Processing Files (1 / 1): 100%|██████████| 1.65MB / 1.65MB,  145kB/s  
New Data Upload: 100%|██████████| 1.65MB / 1.65MB,  145kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.99s/ shards]


Pushed ezstance to https://huggingface.co/datasets/nityaak/ezstance-stance-test-mixed


## Recap

- Same `conversations` format as every other notebook here
- Uses the **test** splits (`mtcsd_test.csv`, `ezstance_test_mixed.csv`), never the
  training files, so there's no overlap with anything the model could have trained on
- Once pushed, `03_colab_hyperparameter_playground.ipynb` can load
  `nityaak/mtcsd-stance-test` / `nityaak/ezstance-stance-test-mixed` the same way it
  already loads `nityaak/semeval-stance-test-gold`
